In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

#1.Load the processed data
train = pd.read_csv('train_processed.csv')
test = pd.read_csv('test_processed.csv')

#2.Separate features and target
X = train.drop('Churn', axis=1)
y = train['Churn']

#3.Create validation Set (10% of training data)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

#4.Initialize the XGBoost Model
xgb_model = xgb.XGBClassifier(
    n_estimators=5000,        #High limit, let early stopping decide the end
    learning_rate=0.05,       #Slow and steady learning
    max_depth=6,              #Standard tree depth
    subsample=0.8,            #Use 80% of data per tree to prevent overfitting
    colsample_bytree=0.8,     #Use 80% of features per tree
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',       #Faster training method
    random_state=42,
    early_stopping_rounds=50  #Stop if AUC doesn't improve for 50 rounds
)

#5.Train the model
print("Starting training...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100 #Print progress every 100 trees
)

#6.Generate predictions
print("Training complete. Generating test predictions...")
xgb_probs = xgb_model.predict_proba(test)[:, 1]

#7.Create Submission
lgbm_ref = pd.read_csv('LGBMsubmission.csv')
submission = pd.DataFrame({
    'id': lgbm_ref['id'],
    'Churn': xgb_probs
})

submission.to_csv('xgb_submission.csv', index=False)
print("XGBoost submission saved")

Starting training...
[0]	validation_0-auc:0.90688
[100]	validation_0-auc:0.91430
[200]	validation_0-auc:0.91583
[300]	validation_0-auc:0.91634
[400]	validation_0-auc:0.91656
[500]	validation_0-auc:0.91671
[600]	validation_0-auc:0.91674
[602]	validation_0-auc:0.91673
Training complete. Generating test predictions...
XGBoost submission saved
